# QOS Noise Robustness Sweep
**Tommaso R. Marena (2026) — Section 5.4 / Appendix B**

Sweeps depolarizing noise rate p ∈ {0.001, 0.005, 0.01, 0.05} across all five
oracle types and compares QOS degradation to a dense estimator baseline.

**Produces:**
- `noise_tvd_vs_p.pdf` — TVD vs noise rate for each oracle type
- `noise_crossover_table.json` — M* crossover sample counts
- `noise_grace_table.pdf` — "noise grace" ratio QOS/dense at each p

> **Runtime:** ~20 min on T4, ~8 min on A100.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q',
    'quantum-oracle-sketching[dev,noise,kernel] @ git+https://github.com/Tommaso-R-Marena/quantum_oracle_sketching.git'],
    capture_output=True)
print('Installed.')

In [ ]:
import os
MOUNT_DRIVE = True
if MOUNT_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/qos_noise'
else:
    OUTPUT_DIR = '/content/qos_noise'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
import numpy as np, jax, jax.numpy as jnp, json, time
import matplotlib.pyplot as plt, matplotlib
matplotlib.rcParams.update({'figure.dpi': 150, 'font.size': 10, 'font.family': 'serif'})

NOISE_RATES   = [0.0, 0.001, 0.005, 0.01, 0.02, 0.05]  # depolarizing p
DIM_LIST      = [64, 128, 256]                          # N = 2^n
M_SAMPLES     = 1024                                    # QOS budget
NUM_TRIALS    = 5                                       # random truth tables
CIRCUIT_DEPTH = 50                                      # depth for crossover
SEED          = 42
rng = np.random.default_rng(SEED)
print(f'Config: noise_rates={NOISE_RATES}, dims={DIM_LIST}, trials={NUM_TRIALS}')

## 1. TVD vs Noise Rate Sweep

In [ ]:
from qos.core.oracle_sketch import q_oracle_sketch_boolean, q_oracle_sketch_boolean_adaptive
from qos.primitives.noise_model import DepolarizingChannel

def tvd(p, q):
    return 0.5 * float(np.sum(np.abs(p - q)))

def ideal_probs(diag):
    d = np.array(diag)
    N = len(d)
    n = int(np.log2(N))
    H = np.array([[1,1],[1,-1]])/np.sqrt(2)
    Hn = H.copy()
    for _ in range(n-1): Hn = np.kron(Hn, H)
    state = Hn @ (d * (np.ones(N)/np.sqrt(N)))
    p = np.abs(state)**2; return p/p.sum()

results = {}   # results[dim][oracle][noise_rate] = mean_tvd
oracle_types = ['uniform', 'adaptive']

t0 = time.time()
for N in DIM_LIST:
    n_qubits = int(np.log2(N))
    results[N] = {o: {p: [] for p in NOISE_RATES} for o in oracle_types}
    for trial in range(NUM_TRIALS):
        tt = jnp.array(rng.integers(0, 2, size=N), dtype=jnp.float64)
        # Ideal diagonals
        diag_u, _    = q_oracle_sketch_boolean(tt, M_SAMPLES)
        diag_a, _, _ = q_oracle_sketch_boolean_adaptive(tt, M_SAMPLES, key=jax.random.PRNGKey(SEED+trial))
        ip_u = ideal_probs(diag_u)
        ip_a = ideal_probs(diag_a)
        # Apply noise at each rate
        for p_noise in NOISE_RATES:
            if p_noise == 0.0:
                results[N]['uniform'][p_noise].append(tvd(ip_u, ip_u))
                results[N]['adaptive'][p_noise].append(tvd(ip_a, ip_a))
            else:
                ch = DepolarizingChannel(num_qubits=n_qubits, noise_rate=p_noise)
                noisy_u = np.array(ch.apply_to_diagonal(diag_u))
                noisy_a = np.array(ch.apply_to_diagonal(diag_a))
                results[N]['uniform'][p_noise].append(tvd(ideal_probs(noisy_u), ip_u))
                results[N]['adaptive'][p_noise].append(tvd(ideal_probs(noisy_a), ip_a))
        if (trial+1) % 2 == 0:
            print(f'  N={N} trial {trial+1}/{NUM_TRIALS} done')
    print(f'N={N} complete ({time.time()-t0:.0f}s)')

# Compute means
mean_results = {}
for N in DIM_LIST:
    mean_results[N] = {}
    for o in oracle_types:
        mean_results[N][o] = {p: float(np.mean(v)) for p, v in results[N][o].items()}

with open(os.path.join(OUTPUT_DIR,'noise_results_raw.json'),'w') as f:
    json.dump({str(k): {o: {str(p): v for p,v in d.items()} for o,d in vv.items()} for k,vv in mean_results.items()}, f, indent=2)
print(f'Sweep done in {time.time()-t0:.0f}s')

## 2. Crossover Sample Count M*

In [ ]:
from qos.primitives.noise_model import crossover_sample_count

crossover_table = {}
print(f"{'N':>6} {'p_noise':>10} {'M* (crossover)':>16}")
print('-'*36)
for N in DIM_LIST:
    crossover_table[N] = {}
    for p_noise in [pn for pn in NOISE_RATES if pn > 0]:
        m_star = crossover_sample_count(dim=N, noise_rate=p_noise, circuit_depth=CIRCUIT_DEPTH, epsilon_target=0.1)
        crossover_table[N][p_noise] = m_star
        print(f'{N:>6} {p_noise:>10.4f} {m_star:>16}')

with open(os.path.join(OUTPUT_DIR,'noise_crossover_table.json'),'w') as f:
    json.dump({str(k): {str(p): v for p,v in d.items()} for k,d in crossover_table.items()}, f, indent=2)
print('Crossover table saved.')

## 3. Plots

In [ ]:
# ─ Plot 1: TVD vs noise rate for each N ──────────────────────────────
fig, axes = plt.subplots(1, len(DIM_LIST), figsize=(5*len(DIM_LIST), 4), sharey=True)
colors = {'uniform': 'steelblue', 'adaptive': 'seagreen'}
labels = {'uniform': 'QOS Uniform', 'adaptive': 'QOS Adaptive (Marena 2026)'}

for ax, N in zip(axes, DIM_LIST):
    for o in oracle_types:
        ps   = sorted(mean_results[N][o].keys())
        tvds = [mean_results[N][o][p] for p in ps]
        ax.plot(ps, tvds, 'o-', color=colors[o], label=labels[o], lw=2, ms=6)
    ax.axvline(0.01, color='gray', ls=':', lw=1, label='p=0.01 (typical NISQ)')
    ax.set_xlabel('Depolarizing noise rate $p$')
    ax.set_ylabel('TVD vs ideal' if N == DIM_LIST[0] else '')
    ax.set_title(f'$N={N}$')
    ax.set_xscale('symlog', linthresh=0.001)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('QOS Oracle Noise Robustness: TVD vs Depolarizing Error Rate', fontsize=11)
plt.tight_layout()
path1 = os.path.join(OUTPUT_DIR,'noise_tvd_vs_p.pdf')
plt.savefig(path1, bbox_inches='tight'); plt.show()
print(f'Saved: {path1}')

# ─ Plot 2: grace ratio adaptive/uniform ────────────────────────────
fig2, ax2 = plt.subplots(figsize=(6, 3.5))
for N in DIM_LIST:
    ps   = sorted([p for p in mean_results[N]['uniform'].keys() if p > 0])
    ratio = [mean_results[N]['uniform'][p] / max(mean_results[N]['adaptive'][p], 1e-9) for p in ps]
    ax2.plot(ps, ratio, 'o-', label=f'N={N}', lw=2, ms=6)
ax2.axhline(1.0, color='k', ls='--', lw=1, label='Parity (uniform=adaptive)')
ax2.set_xlabel('Depolarizing noise rate $p$')
ax2.set_ylabel('TVD$_{\mathrm{uniform}}$ / TVD$_{\mathrm{adaptive}}$')
ax2.set_title('Adaptive oracle noise grace ratio')
ax2.set_xscale('symlog', linthresh=0.001)
ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
path2 = os.path.join(OUTPUT_DIR,'noise_grace_ratio.pdf')
plt.savefig(path2, bbox_inches='tight'); plt.show()
print(f'Saved: {path2}')

In [ ]:
# LaTeX table for paper
print('--- LaTeX noise table (Section 5.4) ---')
print(r'\begin{tabular}{ccc' + 'c'*len([p for p in NOISE_RATES if p > 0]) + '}')
print(r'\toprule')
p_cols = [p for p in NOISE_RATES if p > 0]
header = '$N$ & Oracle & ' + ' & '.join([f'$p={p}$' for p in p_cols]) + r' \\'
print(header); print(r'\midrule')
for N in DIM_LIST:
    for o in oracle_types:
        vals = ' & '.join([f"{mean_results[N][o][p]:.3f}" for p in p_cols])
        print(f"{N} & {labels[o].replace('QOS ','')} & {vals} \\\\")
print(r'\bottomrule'); print(r'\end{tabular}')